# 06 - Add source features to v0

Compare one or two source-derived inputs with all seven additional source features.

| Variant | Features | Additional source information |
| --- | ---: | --- |
| `v0` | 19 | None |
| `v0_src_bound` | 20 | `src_bound_rel` |
| `v0_src_contrast` | 20 | `src_edge_contrast` |
| `v0_src_bound_contrast` | 21 | Both |
| `v1_no_shift` | 26 | All seven source features |

All variants exclude the two merge-shift features. The default fresh run uses seeds 0, 1, and 2:
**15 model fits**. This is an additive predictive-feature experiment; it does not demonstrate that
a feature can be stored as one query-independent scalar or computed without source histograms.

This notebook is self-contained apart from the repository's existing feature/training modules. All outputs are intentionally cleared in Git; running the cells creates the result files.

In [ ]:
%pip install lightgbm==4.6.0

## Settings and input requirements

Run the next cell, then review the widgets. Use the same input files and compute/package environment
across experiments. `data_root/<dataset>/` must contain `queries-Ans.txt` and `queries-meta.csv`;
`sample_root/<dataset>/` must contain `rows.csv`, `src_*.txt`, and the referenced `arm_*.txt` histograms.
Git does not contain those logger exports. Missing files or permissions stop execution.

- `repo_root`: your personal Git folder containing the existing `ml` modules.
- `output_root`: a personal results directory outside the checkout, shared workspace, and input roots.
- `experiment`: your personal `/Users/<account>/...` MLflow experiment.
- `seeds=0,1,2`, `epsilons=0.25`, `delta=0.05`, `max_trees=4000`, and `num_threads=2` match the study.
- `copy_inputs=true` creates input snapshots and records SHA-256 fingerprints.
- `mlflow_enabled=true` logs model artifacts; set false for file-only output.

This notebook trains every selected model from scratch. No previous personal seed-0 run is required.
Each execution creates a unique result directory; previous outputs and team inputs remain unchanged.
Databricks retains old widget values, so review them before continuing. For local execution, the same
settings are accepted as uppercase environment variables. All experiment helpers are defined below;
only the three existing `features`, `features_v1`, and `gate_eval` modules are imported from the repo.

In [ ]:
STUDY = 'source-feature'
NOTEBOOK_FILENAME = '06_source_feature_ablation.ipynb'
DEFAULT_VARIANTS = 'v0,v0_src_bound,v0_src_contrast,v0_src_bound_contrast,v1_no_shift'
BASELINE_VARIANT = 'v0'

import hashlib
import importlib
import importlib.metadata
import inspect
import json
import math
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import time
from uuid import uuid4

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def _repo_root():
    try:
        notebook = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        root = notebook.split('/ml/notebooks/')[0]
        return root if root.startswith('/Workspace/') else '/Workspace' + root
    except Exception:
        return os.environ.get('REPO_ROOT', os.getcwd())


def _defaults():
    repo = _repo_root()
    output, experiment = '', ''
    parts = Path(repo).parts
    if 'Users' in parts and len(parts) > parts.index('Users') + 1:
        user = parts.index('Users') + 1
        output = str(Path(*parts[:user + 1]) / 'synopsislake-results')
        experiment = f'/Users/{parts[user]}/synopsislake-{STUDY}'
    volume = os.environ.get('VOLUME', '/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake')
    return dict(repo_root=repo, data_root=volume + '/data/ml', sample_root=volume + '/output/ml',
                output_root=output, experiment=experiment, dataset='authors_range', seeds='0,1,2',
                variants=DEFAULT_VARIANTS, epsilons='0.25', delta='0.05', max_trees='4000',
                num_threads='2', copy_inputs='true', mlflow_enabled='true')


DEFAULTS = _defaults()
try:
    for key, value in DEFAULTS.items():
        dbutils.widgets.text(key, value)
    get = dbutils.widgets.get
except NameError:
    def get(name):
        return os.environ.get(name.upper(), DEFAULTS[name])

repo = Path(get('repo_root')).resolve()
for filename in ('features.py', 'features_v1.py', 'gate_eval.py'):
    if not (repo / 'ml' / filename).is_file():
        raise FileNotFoundError(f'Check repo_root: missing existing module {repo / "ml" / filename}')
sys.path.insert(0, str(repo / 'ml'))
for name in ('features', 'features_v1', 'gate_eval'):
    sys.modules.pop(name, None)
importlib.invalidate_caches()
from features import FEATURES, check_estimates
from features_v1 import FEATURES_V1, GROUPS, build_table_v1, check_source_estimates
from gate_eval import PARAMS, baseline_scores, cutoff, fit, split_ids

## Feature definitions

Preserve the existing feature order. Each seed uses the same query-ID split across variants.

In [ ]:
def variants():
    return {
        'v0': list(FEATURES),
        'v0_src_bound': FEATURES + ['src_bound_rel'],
        'v0_src_contrast': FEATURES + ['src_edge_contrast'],
        'v0_src_bound_contrast': FEATURES + ['src_bound_rel', 'src_edge_contrast'],
        'v1_no_shift': [name for name in FEATURES_V1 if name not in GROUPS['shift']],
    }

## Input, split, and artifact helpers

Inputs are read-only. Outputs are restricted to a unique personal directory.

In [ ]:
def _bool(value):
    if isinstance(value, bool):
        return value
    if str(value).lower() not in ("true", "false"):
        raise ValueError("boolean settings must be true or false")
    return str(value).lower() == "true"


def _jsonable(value):
    if isinstance(value, dict):
        return {str(k): _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, Path):
        return str(value)
    return value


def _write_json(path, value):
    Path(path).write_text(json.dumps(_jsonable(value), indent=2, allow_nan=False), encoding="utf-8")


def _hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def validate_output_root(output_root, repo_root, data_root, sample_root):
    if not str(output_root).strip():
        raise ValueError("Set output_root to a personal results directory before running")
    output = Path(output_root).expanduser().resolve()
    if output.is_relative_to(Path("/Workspace/Shared")):
        raise ValueError("output_root must be personal, not /Workspace/Shared")
    for protected in (repo_root, data_root, sample_root):
        if output.is_relative_to(Path(protected).resolve()):
            raise ValueError("output_root must be outside the checkout and input directories")
    return output


def make_split_manifest(df, seed):
    train, cal, test = split_ids(df.qId, seed)
    early = set(sorted(train)[::10])
    return pd.DataFrame([
        {"seed": seed, "qId": int(qid), "split": split, "early_stopping": qid in early}
        for split, ids in (("train", train), ("calibration", cal), ("test", test))
        for qid in sorted(ids)
    ])


def _auc(error, score, eps):
    bad = np.asarray(error) > eps
    return float(roc_auc_score(bad, score)) if len(np.unique(bad)) == 2 else np.nan


def _inputs(data_dir, run_dir):
    files = [data_dir / "queries-Ans.txt", data_dir / "queries-meta.csv", run_dir / "rows.csv"]
    for path in files:
        if not path.is_file():
            raise FileNotFoundError(f"Missing input: {path}. Git clone does not create logger outputs.")
    sources = sorted(run_dir.glob("src_*.txt"))
    if not sources:
        raise FileNotFoundError(f"Missing source histogram exports: {run_dir}/src_*.txt")
    rows = pd.read_csv(run_dir / "rows.csv", usecols=["method", "lonRes"])
    resolutions = rows.loc[rows.method == "ARM", "lonRes"].unique()
    if not len(resolutions):
        raise ValueError("rows.csv has no ARM records")
    files += sources + [run_dir / f"arm_{int(res)}.txt" for res in sorted(resolutions)]
    for path in files:
        if not path.is_file():
            raise FileNotFoundError(f"Missing histogram export: {path}")
    return files


def _snapshot(files, data_dir, run_dir, destination, copy_inputs):
    manifest = []
    for source in files:
        before = source.stat()
        target = source
        if copy_inputs:
            kind = "data" if source.parent == data_dir else "run"
            target = destination / "inputs" / kind / source.name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source, target)
        checksum = _hash(target)
        after = source.stat()
        if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):
            raise RuntimeError(f"Input changed during snapshot: {source}")
        manifest.append(dict(source=str(source), used_path=str(target), size_bytes=before.st_size, sha256=checksum))
    return manifest

## Calibration and quality accounting

Use the existing calibration-only cutoff. Keep accepted and violating counts, and leave zero-acceptance risk undefined.

In [ ]:
def evaluate_scores(cal, test, cal_scores, test_scores, epsilons=(0.25,), delta=0.05):
    """Same empirical cutoff as gate_eval; count cases and leave empty risk undefined."""
    details, summaries = [], []
    if not len(cal) or not len(test):
        raise ValueError("calibration and test cohorts must both be nonempty")
    for name, test_score in test_scores.items():
        sc, st = np.asarray(cal_scores[name]), np.asarray(test_score)
        if len(sc) != len(cal) or len(st) != len(test) or not np.isfinite(sc).all() or not np.isfinite(st).all():
            raise ValueError("scores must be finite and match calibration/test rows")
        for eps in epsilons:
            start = len(details)
            scopes = [("pooled", None, np.ones(len(cal), bool), np.ones(len(test), bool))]
            scopes += [("per_rate", float(rate), cal.mergeRate.to_numpy() == rate,
                        test.mergeRate.to_numpy() == rate) for rate in sorted(test.mergeRate.unique())]
            for scope, rate, c, t in scopes:
                ec, et = cal.relErr.to_numpy()[c], test.relErr.to_numpy()[t]
                threshold = float(cutoff(sc[c], ec, eps, delta)) if c.any() else -np.inf
                ac = sc[c] <= threshold
                at = st[t] <= threshold
                count, bad = int(at.sum()), int(((et > eps) & at).sum())
                risk = bad / count if count else np.nan
                details.append(dict(
                    score=name, eps=eps, scope=scope, mergeRate=rate,
                    cutoff=threshold if math.isfinite(threshold) else None,
                    accept_none=not math.isfinite(threshold),
                    calibration_count=int(c.sum()), calibration_accepted_count=int(ac.sum()),
                    calibration_violation_count=int(((ec > eps) & ac).sum()),
                    test_count=int(t.sum()), accepted_count=count, violation_count=bad,
                    coverage=count / int(t.sum()), violation=risk,
                    quality_target_met=(risk <= delta) if count else None,
                    auc=_auc(et, st[t], eps),
                ))
            group = pd.DataFrame(details[start:])
            pooled, rates = group[group.scope == "pooled"].iloc[0], group[group.scope == "per_rate"]
            n, bad = int(rates.accepted_count.sum()), int(rates.violation_count.sum())
            summaries.append(dict(
                score=name, eps=eps, auc=pooled.auc, coverage=pooled.coverage,
                violation=pooled.violation, accepted_count=int(pooled.accepted_count),
                violation_count=int(pooled.violation_count),
                auc_per_rate=rates.auc.mean(), coverage_per_rate=rates.coverage.mean(),
                violation_per_rate=rates.violation.mean(), rates_with_acceptances=int((rates.accepted_count > 0).sum()),
                rates_total=len(rates), accepted_cases_per_rate=n, violating_cases_per_rate=bad,
                violation_pooled_per_rate_gates=bad / n if n else np.nan,
                unique_test_queries=int(test.qId.nunique()), test_cases=len(test),
            ))
    return pd.DataFrame(summaries), pd.DataFrame(details)

## Train and save a model

Use the existing LightGBM fit and split functions. Early stopping uses training queries only; calibration selects each cutoff and test only evaluates it.

In [ ]:
def _train(df, features, seed, params, train_mask=None, test_mask=None):
    tr_ids, ca_ids, te_ids = split_ids(df.qId, seed)
    train_mask = pd.Series(True, index=df.index) if train_mask is None else train_mask
    test_mask = pd.Series(True, index=df.index) if test_mask is None else test_mask
    train = df[df.qId.isin(tr_ids) & train_mask]
    cal = df[df.qId.isin(ca_ids) & test_mask]
    test = df[df.qId.isin(te_ids) & test_mask]
    if train.qId.nunique() < 3 or not len(cal) or not len(test):
        raise ValueError("Not enough query IDs in train/calibration/test; do not resplit to fix this")
    start = time.perf_counter()
    model = fit(train, features, seed, params=params)
    fit_seconds = time.perf_counter() - start
    start = time.perf_counter()
    sc = {"learned": model.predict(cal[features]), **baseline_scores(cal)}
    st = {"learned": model.predict(test[features]), **baseline_scores(test)}
    return model, train, cal, test, sc, st, fit_seconds, time.perf_counter() - start


def _record_model(df, name, features, seed, params, epsilons, delta, folder, train_mask=None, test_mask=None):
    model, train, cal, test, sc, st, fit_seconds, predict_seconds = _train(
        df, features, seed, params, train_mask, test_mask)
    start = time.perf_counter()
    summary, details = evaluate_scores(cal, test, sc, st, epsilons, delta)
    evaluate_seconds = time.perf_counter() - start
    folder.mkdir(parents=True, exist_ok=False)
    model.booster_.save_model(str(folder / "model.txt"))
    _write_json(folder / "schema.json", dict(
        features=features, variant=name, seed=seed, params=params,
        best_iteration=model.best_iteration_, min_exact=10,
        label="log(max(abs(estimate-truth)/truth, 0.001))",
        score="predicted log-error; lower is better; thresholds are in score space",
    ))
    _write_json(folder / "cutoffs.json", details[details.score == "learned"].to_dict("records"))
    predictions = test[["qId", "mergeRate", "relErr"]].copy()
    predictions["score"] = st["learned"]
    predictions.to_csv(folder / "test_predictions.csv", index=False)
    calibration_predictions = cal[["qId", "mergeRate", "relErr"]].copy()
    calibration_predictions["score"] = sc["learned"]
    calibration_predictions.to_csv(folder / "calibration_predictions.csv", index=False)
    # Export effective row cohorts as well as the common global query split.
    early = set(sorted(train.qId.unique())[::10])
    cohorts = pd.concat([part[["qId", "mergeRate"]].assign(split=split)
                         for split, part in (("train", train), ("calibration", cal), ("test", test))], ignore_index=True)
    cohorts["early_stopping"] = (cohorts.split == "train") & cohorts.qId.isin(early)
    cohorts.to_csv(folder / "cohorts.csv", index=False)
    for frame in (summary, details):
        frame.insert(0, "variant", name)
        frame.insert(1, "seed", seed)
        frame.insert(2, "n_features", len(features))
    summary["fit_seconds"], summary["prediction_seconds"] = fit_seconds, predict_seconds
    summary["evaluation_seconds"], summary["best_iteration"] = evaluate_seconds, model.best_iteration_
    importance = pd.DataFrame(dict(feature=features, gain=model.booster_.feature_importance("gain"), variant=name, seed=seed))
    summary.to_csv(folder / "results.csv", index=False)
    importance.to_csv(folder / "importance.csv", index=False)
    return summary, details, importance


def _log_mlflow(mlflow, folder, result, config, features):
    with mlflow.start_run(run_name=f"{config['dataset']}-{folder.parent.name}-{folder.name}"):
        mlflow.log_params(dict(dataset=config["dataset"], variant=folder.parent.name,
                              seed=folder.name.removeprefix("seed_"),
                              features=",".join(features), **config["model_params"]))
        mlflow.set_tag("personal_result_dir", str(folder))
        for row in result.itertuples():
            for field in ("auc_per_rate", "coverage_per_rate", "violation_per_rate", "fit_seconds"):
                value = float(getattr(row, field))
                if math.isfinite(value):
                    mlflow.log_metric(f"{row.score}_eps{row.eps}_{field}", value)
        # Keep the data snapshot local; log only the model and its evaluation artifacts.
        mlflow.log_artifacts(str(folder), artifact_path="model_evaluation")

## Run and summarize the study

The reference for paired deltas is `v0`. No previous experiment outputs or saved model are required.

In [ ]:
def _tables(results, details, destination, delta):
    learned = results[results.score == 'learned']
    metrics = ['auc_per_rate', 'coverage_per_rate', 'violation_per_rate',
               'violation_pooled_per_rate_gates', 'fit_seconds']
    grouped = learned.groupby(['eps', 'variant'])
    summary = grouped[metrics].agg(['mean', 'std'])
    summary.columns = [f'{metric}_{stat}' for metric, stat in summary.columns]
    summary = summary.reset_index().merge(grouped.seed.nunique().rename('n_seed').reset_index())
    for metric in metrics:
        summary = summary.merge(grouped[metric].count().rename(metric + '_n').reset_index())
    counts = grouped[['accepted_cases_per_rate', 'violating_cases_per_rate']].sum().reset_index()
    counts['case_weighted_violation'] = counts.violating_cases_per_rate / counts.accepted_cases_per_rate.where(counts.accepted_cases_per_rate > 0)
    summary = summary.merge(counts, on=['eps', 'variant'], validate='one_to_one')
    summary.to_csv(destination / 'summary.csv', index=False)
    rate = details[(details.score == 'learned') & (details.scope == 'per_rate')].copy()
    rate['above_target'] = (rate.accepted_count > 0) & (rate.violation > delta)
    rates = rate.groupby(['eps', 'variant', 'mergeRate']).agg(
        n_seed=('seed', 'nunique'), coverage_mean=('coverage', 'mean'), coverage_std=('coverage', 'std'),
        violation_mean=('violation', 'mean'), violation_std=('violation', 'std'),
        defined_risk_seeds=('violation', 'count'), above_target_seeds=('above_target', 'sum'),
        accepted_cases=('accepted_count', 'sum'), violating_cases=('violation_count', 'sum'),
    ).reset_index()
    rates.to_csv(destination / 'per_rate_summary.csv', index=False)
    paired_metrics = ['coverage_per_rate', 'violation_per_rate', 'auc_per_rate',
                      'violation_pooled_per_rate_gates', 'accepted_cases_per_rate', 'violating_cases_per_rate']
    baseline = learned[learned.variant == BASELINE_VARIANT][['seed', 'eps', *paired_metrics]].rename(
        columns={metric: 'baseline_' + metric for metric in paired_metrics})
    paired = learned[learned.variant != BASELINE_VARIANT].merge(baseline, on=['seed', 'eps'], validate='many_to_one')
    for metric in paired_metrics:
        paired['delta_' + metric] = paired[metric] - paired['baseline_' + metric]
    paired.to_csv(destination / 'paired_deltas.csv', index=False)
    return summary, rates, paired


def plan_experiment(config):
    cfg = dict(config)
    dataset = str(cfg['dataset']).strip()
    if not dataset or dataset in ('.', '..') or '/' in dataset or '\\' in dataset:
        raise ValueError('dataset must be a single directory name')
    cfg['dataset'] = dataset
    validate_output_root(cfg['output_root'], cfg['repo_root'], cfg['data_root'], cfg['sample_root'])
    names = [name.strip() for name in cfg['variants'].split(',')]
    if not names or len(names) != len(set(names)) or set(names) - variants().keys() or BASELINE_VARIANT not in names:
        raise ValueError(f'Select distinct variants including {BASELINE_VARIANT} from {list(variants())}')
    seeds = [int(value.strip()) for value in str(cfg['seeds']).split(',')]
    epsilons = [float(value.strip()) for value in str(cfg['epsilons']).split(',')]
    delta = float(cfg['delta'])
    if not seeds or len(set(seeds)) != len(seeds) or min(seeds) < 0:
        raise ValueError('seeds must be distinct nonnegative integers')
    if not epsilons or len(set(epsilons)) != len(epsilons) or any(not math.isfinite(e) or e <= 0 for e in epsilons) or not 0 <= delta < 1:
        raise ValueError('Invalid error or violation target')
    cfg['model_params'] = dict(PARAMS, n_estimators=int(cfg['max_trees']), n_jobs=int(cfg['num_threads']))
    if cfg['model_params']['n_estimators'] < 1 or cfg['model_params']['n_jobs'] < 1:
        raise ValueError('max_trees and num_threads must be positive')
    _bool(cfg['copy_inputs'])
    if _bool(cfg['mlflow_enabled']) and not str(cfg['experiment']).startswith('/Users/'):
        raise ValueError('Set a personal /Users/<account>/... MLflow experiment')
    _inputs(Path(cfg['data_root']).resolve() / dataset, Path(cfg['sample_root']).resolve() / dataset)
    return dict(config=cfg, variants=names, seeds=seeds, epsilons=epsilons, delta=delta,
                fit_count=len(names) * len(seeds))


def run_experiment(config):
    started = time.perf_counter()
    plan = plan_experiment(config)
    cfg = plan['config']
    mlflow = None
    if _bool(cfg['mlflow_enabled']):
        import mlflow
        mlflow.set_experiment(cfg['experiment'])
    data = Path(cfg['data_root']).resolve() / cfg['dataset']
    samples = Path(cfg['sample_root']).resolve() / cfg['dataset']
    destination = Path(cfg['output_root']).resolve() / f"{cfg['dataset']}-{STUDY}-{uuid4().hex[:12]}"
    destination.mkdir(parents=True, exist_ok=False)
    state = dict(status='running', study=STUDY, config=cfg, timings={},
                 feature_definitions={name: variants()[name] for name in plan['variants']},
                 notes=['Fresh training; no previous personal results or models are required.',
                        'Offline quality study, not end-to-end latency or a source-I/O saving.',
                        'Exact answers below 10 are excluded; zero-count quality is not evaluated.',
                        'Empirical calibration does not guarantee 5% test risk; zero acceptance has undefined risk.',
                        'Query splits and merge rates overlap; seed SDs are descriptive, not confidence intervals.'])
    try:
        tick = time.perf_counter()
        manifest = _snapshot(_inputs(data, samples), data, samples, destination, _bool(cfg['copy_inputs']))
        _write_json(destination / 'input_manifest.json', manifest)
        state['input_sha256'] = {('data/' if Path(row['source']).parent == data else 'run/') + Path(row['source']).name: row['sha256'] for row in manifest}
        state['timings']['input_preparation_seconds'] = time.perf_counter() - tick
        if _bool(cfg['copy_inputs']):
            data, samples = destination / 'inputs/data', destination / 'inputs/run'
        tick = time.perf_counter()
        frame = build_table_v1(str(samples), str(data))
        if not len(frame) or frame.duplicated(['qId', 'mergeRate']).any():
            raise ValueError('Missing eligible rows or duplicate query/histogram cases')
        if not np.isfinite(frame[FEATURES_V1 + ['label', 'relErr', 'mergeRate']].to_numpy(float)).all():
            raise ValueError('Nonfinite features or labels; check inputs')
        state.update(rows=len(frame), unique_queries=int(frame.qId.nunique()),
                     dropped_small_exact=frame.attrs.get('dropped_small_exact', 0),
                     estimate_gaps=dict(merged=check_estimates(frame), sources=check_source_estimates(frame)))
        state['timings']['feature_build_seconds'] = time.perf_counter() - tick
        state['package_versions'] = {name: importlib.metadata.version(name) for name in
                                     ('lightgbm', 'numpy', 'pandas', 'scipy', 'scikit-learn')}
        state['python'] = platform.python_version()
        state['code_sha256'] = {name: _hash(repo / 'ml' / name) for name in ('features.py', 'features_v1.py', 'gate_eval.py')}
        state['inline_helper_sha256'] = {}
        for function in (variants, _snapshot, evaluate_scores, _train, _record_model,
                         _tables, plan_experiment, run_experiment):
            try:
                state['inline_helper_sha256'][function.__name__] = hashlib.sha256(inspect.getsource(function).encode()).hexdigest()
            except (OSError, TypeError) as error:
                state.setdefault('inline_helper_hash_unavailable', {})[function.__name__] = type(error).__name__
        # Databricks may expose notebooks as workspace objects, not ordinary files.
        try:
            state['code_sha256'][NOTEBOOK_FILENAME] = _hash(repo / 'ml/notebooks' / NOTEBOOK_FILENAME)
        except OSError as error:
            state['notebook_hash_unavailable'] = type(error).__name__
        try:
            state['git_commit'] = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True, stderr=subprocess.DEVNULL).strip()
            state['git_dirty'] = bool(subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain'], text=True, stderr=subprocess.DEVNULL))
        except (OSError, subprocess.CalledProcessError):
            state['git_commit'] = None
        pd.concat([make_split_manifest(frame, seed) for seed in plan['seeds']], ignore_index=True).to_csv(destination / 'splits.csv', index=False)
        _write_json(destination / 'run.json', state)
        results, details, importances = [], [], []
        for name in plan['variants']:
            for seed in plan['seeds']:
                folder = destination / 'models' / name / f'seed_{seed}'
                result, gates, importance = _record_model(frame, name, variants()[name], seed,
                                                          cfg['model_params'], plan['epsilons'], plan['delta'], folder)
                results.append(result)
                details.append(gates)
                importances.append(importance)
                if mlflow is not None:
                    _log_mlflow(mlflow, folder, result, cfg, variants()[name])
                print(f'{name} seed={seed}: trained; saved {folder}')
        combined = pd.concat(results, ignore_index=True)
        gates = pd.concat(details, ignore_index=True)
        combined.to_csv(destination / 'by_seed.csv', index=False)
        gates.to_csv(destination / 'gate_details.csv', index=False)
        pd.concat(importances, ignore_index=True).to_csv(destination / 'importance.csv', index=False)
        _tables(combined, gates, destination, plan['delta'])
        state['timings']['current_fit_seconds'] = float(combined[(combined.score == 'learned') & (combined.eps == plan['epsilons'][0])].fit_seconds.sum())
        state['timings']['experiment_wall_seconds'] = time.perf_counter() - started
        state['status'] = 'complete'
        _write_json(destination / 'run.json', state)
        print('COMPLETE:', destination)
        return destination
    except Exception as error:
        state.update(status='failed', error=f'{type(error).__name__}: {error}')
        _write_json(destination / 'run.json', state)
        raise

## Review the training plan

Review paths, features, and fit count before running the next training cell. The default study performs **15 fresh fits**.

In [ ]:
CONFIG = {key: get(key) for key in DEFAULTS}
PLAN = plan_experiment(CONFIG)
print('Repo:', repo)
print('Read-only input roots:', CONFIG['data_root'], CONFIG['sample_root'])
print('Personal output root:', CONFIG['output_root'])
print('Seeds:', PLAN['seeds'], 'new model fits:', PLAN['fit_count'])
print(pd.DataFrame([dict(variant=name, features=len(variants()[name])) for name in PLAN['variants']]).to_string(index=False))

## Execute

This cell trains the selected models from the actual inputs. Missing data or permissions fail clearly. Re-running creates another result directory.

In [ ]:
if {key: get(key) for key in DEFAULTS} != CONFIG:
    raise ValueError('Settings changed; rerun Review the training plan before training')
result_dir = run_experiment(CONFIG)
print('Results for this study:', result_dir)

## Read the results

`summary.csv` reports learned-model means and sample standard deviations across seeds.
`by_seed.csv` contains the individual results. `gate_details.csv` records pooled/per-rate cutoffs,
calibration/test sample counts, acceptances, violations, and observed target status.
`per_rate_summary.csv` records defined-risk seed counts and target exceedances per merge rate.
`paired_deltas.csv` compares each alternative with the reference at the same seed and epsilon;
deltas are alternative minus reference. Compare coverage and risk together.

Every model folder contains `model.txt`, `schema.json`, `cutoffs.json`, `cohorts.csv`,
`test_predictions.csv`, `calibration_predictions.csv`, `results.csv`, and `importance.csv`.
Cutoffs are in predicted log-error space, not relative-error units. `run.json` must have
`status=complete` before the full study is considered complete. Record runtime and package versions.

Zero accepted answers have undefined risk, not successful 0% violation. A single-class AUC is undefined.
`oracle` uses true error and is an offline reference, not a deployable gate. The 5% target is an empirical
calibration criterion, not a guarantee on test data. Do not tune delta using observed test results.
Queries overlap across merge rates and seeds; standard deviations are descriptive, not confidence intervals.
Previously examined test outcomes are exploratory; use a reserved workload for final confirmation.
All source features are still built, so this study does not measure source-I/O, memory, exact-fallback,
or end-to-end query-latency savings. Timing columns report offline preparation, fitting, prediction,
and evaluation costs. Exact counts below 10 are excluded; zero-count query quality is not evaluated.

In [ ]:
metadata = json.loads((result_dir / 'run.json').read_text())
if metadata['status'] != 'complete':
    raise ValueError('Study is incomplete; inspect run.json')
summary = pd.read_csv(result_dir / 'summary.csv')
deltas = pd.read_csv(result_dir / 'paired_deltas.csv')
rates = pd.read_csv(result_dir / 'per_rate_summary.csv')
try:
    display(summary)
    display(deltas)
    display(rates)
except NameError:
    print(summary.to_string(index=False))
    print(deltas.to_string(index=False))
print('Result files:', result_dir)
print('Share summary.csv, by_seed.csv, paired_deltas.csv, per_rate_summary.csv, gate_details.csv and run.json for analysis.')